## Multi-Agent Weather & Search System

**Challenge 3**: demonstrates the end-to-end implementation of an intelligent multi-agent system built using Google Cloud's Vertex AI Agent Development Kit (ADK) and Reasoning Engines.

---

**Key Architecture & Highlights**

* **Hierarchical Multi-Agent Orchestration:**  
  A coordinator root agent (`main_agent`) evaluates user intent and dynamically delegates queries across specialized sub-agents:
  * **Weather Specialist (`weather_agent`):** Converts US city names to geographic coordinates and queries the National Weather Service (NWS) API for real-time forecasts and severe weather alerts.
  * **Search Specialist (`google_search_agent`):** Utilizes the built-in Google Search tool via `AgentTool` to handle general knowledge, news, and external queries.

* **Lifecycle Callbacks & Safety Filtering:**  
  * **`before_model_callback` (Input Validation & Logging):** Intercepts user prompts to enforce geographical boundaries (ensuring queries target supported US regions) and filters malicious inputs prior to model invocation. It also logs incoming user queries for telemetry.
  * **`after_model_callback` (Response Logging):** Captures and logs all generated LLM responses for monitoring and debugging.

* **Session Management & Local Verification:**  
  * Integrated with `AdkApp` and `InMemoryRunner` for local notebook testing.
  * Supports event streaming (`stream_query`) to verify routing, tool calls, and final responses across diverse test cases.

## Step 1: Install Dependencies

In [1]:
%pip install "google-adk[extensions]" litellm google-genai requests python-dotenv nest-asyncio -q


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Step 2: Import Libraries

In [2]:
import os
import json
import requests
import asyncio
import random
import uuid
from typing import Dict, Any, Optional

# Enable nested event loops for Jupyter
import nest_asyncio
nest_asyncio.apply()

# Google ADK imports
from google.adk.agents.llm_agent import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import InMemoryRunner
from google.adk.tools import AgentTool, google_search
from google.genai.types import Content, Part

import vertexai
from vertexai.preview import reasoning_engines

from dotenv import load_dotenv
load_dotenv()

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


## Step 3: Configuration

Set your API keys here (optional - notebook works without them for common cities)

In [3]:
# API Keys
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY", "")
GOOGLE_MAPS_API_KEY = os.environ.get("GOOGLE_MAPS_API_KEY", "")
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")

# Project configuration
PROJECT_ID = "qwiklabs-gcp-02-138827e82db5"
LOCATION = "us-central1"

# Set environment variables for Vertex AI / Google GenAI SDK
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION

# Initialize Vertex AI globally
vertexai.init(project=PROJECT_ID, location=LOCATION)

print("✅ Configuration loaded and Vertex AI initialized")

✅ Configuration loaded and Vertex AI initialized


## Step 4: Tool 1 - National Weather Service API

Retrieves weather data using latitude and longitude coordinates.
Follows PEP 8 style with type hints and comprehensive docstrings.

In [4]:
def get_weather_by_coordinates(latitude: float, longitude: float) -> Dict[str, Any]:
    """
    Retrieve current weather data from the National Weather Service API.
    
    Args:
        latitude: Latitude coordinate in decimal degrees (-90.0 to 90.0)
        longitude: Longitude coordinate in decimal degrees (-180.0 to 180.0)
    
    Returns:
        Dictionary with weather data:
        - status: 'success' or 'error'
        - temperature: Temperature in Fahrenheit
        - conditions: Weather conditions
        - wind_speed: Wind speed
        - location: Location name
    
    Example:
        >>> weather = get_weather_by_coordinates(37.7749, -122.4194)
        >>> print(weather['temperature'])
        62
    """
    try:
        # Get forecast grid endpoint
        points_url = f"https://api.weather.gov/points/{latitude},{longitude}"
        headers = {
            'User-Agent': 'WeatherAgent/1.0 (Educational)',
            'Accept': 'application/json'
        }
        
        points_response = requests.get(points_url, headers=headers, timeout=10)
        points_response.raise_for_status()
        points_data = points_response.json()
        
        # Get forecast
        forecast_url = points_data['properties']['forecast']
        forecast_response = requests.get(forecast_url, headers=headers, timeout=10)
        forecast_response.raise_for_status()
        forecast_data = forecast_response.json()
        
        # Extract current period
        current = forecast_data['properties']['periods'][0]
        location = points_data['properties']['relativeLocation']['properties']
        
        return {
            'status': 'success',
            'location': f"{location['city']}, {location['state']}",
            'temperature': current['temperature'],
            'temperature_unit': current['temperatureUnit'],
            'conditions': current['shortForecast'],
            'detailed_forecast': current['detailedForecast'],
            'wind_speed': current['windSpeed'],
            'wind_direction': current['windDirection']
        }
        
    except Exception as e:
        return {'status': 'error', 'error': str(e)}

print("✅ Weather function created")

✅ Weather function created


## Step 5: Tool 2 - Google Maps Geocoding API

Converts location names to coordinates. Includes fallback for common cities.

In [5]:
def geocode_location(location: str, api_key: Optional[str] = None) -> Dict[str, Any]:
    """
    Convert location name to latitude and longitude coordinates.
    
    Args:
        location: City name or address to geocode
        api_key: Optional Google Maps API key
    
    Returns:
        Dictionary with coordinates:
        - status: 'success' or 'error'
        - latitude: Latitude coordinate
        - longitude: Longitude coordinate
        - formatted_address: Full address
    
    Example:
        >>> coords = geocode_location('San Francisco, CA')
        >>> print(coords['latitude'], coords['longitude'])
        37.7749 -122.4194
    """
    # Fallback coordinates for common US cities
    CITY_COORDS = {
        'san francisco, ca': {'lat': 37.7749, 'lng': -122.4194},
        'new york city, ny': {'lat': 40.7128, 'lng': -74.0060},
        'chicago, il': {'lat': 41.8781, 'lng': -87.6298},
        'miami, fl': {'lat': 25.7617, 'lng': -80.1918},
        'seattle, wa': {'lat': 47.6062, 'lng': -122.3321},
        'austin, tx': {'lat': 30.2672, 'lng': -97.7431},
        'los angeles, ca': {'lat': 34.0522, 'lng': -118.2437}
    }
    
    location_key = location.lower().strip()
    
    # Try fallback first
    if location_key in CITY_COORDS:
        coords = CITY_COORDS[location_key]
        return {
            'status': 'success',
            'latitude': coords['lat'],
            'longitude': coords['lng'],
            'formatted_address': location,
            'source': 'fallback'
        }
    
    # Try Google Maps API if key provided
    maps_key = api_key or GOOGLE_MAPS_API_KEY
    if maps_key:
        try:
            url = 'https://maps.googleapis.com/maps/api/geocode/json'
            response = requests.get(url, params={'address': location, 'key': maps_key}, timeout=10)
            data = response.json()
            
            if data['status'] == 'OK':
                result = data['results'][0]
                loc = result['geometry']['location']
                return {
                    'status': 'success',
                    'latitude': loc['lat'],
                    'longitude': loc['lng'],
                    'formatted_address': result['formatted_address'],
                    'source': 'google_maps'
                }
        except Exception as e:
            pass
    
    return {
        'status': 'error',
        'error': f'Location not found. Available cities: {list(CITY_COORDS.keys())}'
    }

print("✅ Geocoding function created")

✅ Geocoding function created


## Step 6: Test Individual Functions

In [6]:
# Test geocoding
print("Testing Geocoding:")
coords = geocode_location("San Francisco, CA")
print(json.dumps(coords, indent=2))

# Test weather
print("\nTesting Weather:")
if coords['status'] == 'success':
    weather = get_weather_by_coordinates(coords['latitude'], coords['longitude'])
    print(json.dumps(weather, indent=2))

Testing Geocoding:
{
  "status": "success",
  "latitude": 37.7749,
  "longitude": -122.4194,
  "formatted_address": "San Francisco, CA",
  "source": "fallback"
}

Testing Weather:
{
  "status": "success",
  "location": "San Francisco, CA",
  "temperature": 68,
  "temperature_unit": "F",
  "conditions": "Partly Sunny",
  "detailed_forecast": "Partly sunny. High near 68, with temperatures falling to around 66 in the afternoon. West wind 7 to 15 mph, with gusts as high as 22 mph.",
  "wind_speed": "7 to 15 mph",
  "wind_direction": "W"
}


## Step 7: Create ADK Agent with Tools

In [7]:
# 1. Choose model target
ACTIVE_MODEL = "gemini_flash"  # Options: "gemini_flash" or "claude_sonnet"

# 2. Model Registry
MODEL_REGISTRY = {
    "gemini_flash": "gemini-2.5-flash",
    "claude_sonnet": LiteLlm(model="anthropic/claude-3-5-sonnet-20241022"),
}
SELECTED_MODEL = MODEL_REGISTRY[ACTIVE_MODEL]

# 3. Create Sub-Agents
search_agent = Agent(
    name="google_search_agent",
    model=SELECTED_MODEL,
    description="Agent that searches Google for up-to-date information, news, events, and general queries.",
    instruction="""You are a helpful search assistant. Use the google_search tool to find accurate and up-to-date information.""",
    tools=[google_search],
)

weather_agent = Agent(
    name="weather_assistant",
    model=SELECTED_MODEL,
    description="Weather assistant providing real-time weather info for US locations",
    instruction="""You are a helpful weather assistant.

When users ask about weather:
1. Use geocode_location to convert city name to coordinates
2. Use get_weather_by_coordinates to get weather data
3. Provide a clear, friendly summary
4. Alert on extreme conditions (temp >95°F or <32°F, high winds >25mph)

Be concise and informative.""",
    tools=[geocode_location, get_weather_by_coordinates],
)

# 4. Create Root Agent
MAIN_AGENT_INSTRUCTIONS = """You are the coordinator root agent.
Your job is to assist users by delegating to specialized agents:
- For weather forecasts and current conditions in the US, delegate to weather_assistant.
- For general knowledge, news, facts, or external information, delegate to google_search_agent.
- Synthesize responses clearly for the user."""

main_agent = Agent(
    name="main_agent",
    model=SELECTED_MODEL,
    description="Provides Answers to Users Questions.",
    instruction=MAIN_AGENT_INSTRUCTIONS,
    tools=[AgentTool(agent=search_agent)],
    sub_agents=[weather_agent],
)

# 5. Wrap the ROOT agent into AdkApp and InMemoryRunner
app = reasoning_engines.AdkApp(
    agent=main_agent,
)

runner = InMemoryRunner(
    agent=main_agent, app_name=f"Multi-Agent Coordinator ({ACTIVE_MODEL})"
)

print(
    f"✅ Multi-agent system created using {ACTIVE_MODEL} (AdkApp & Runner pointing to main_agent)"
)

App "Multi-Agent Coordinator (gemini_flash)" can transfer between agents but has no context_cache_config. Every transfer swaps the system instruction and the tool set, so the request prefix changes and the whole prompt is re-sent uncached after each transfer. Set context_cache_config on the app to give each agent its own cache.


✅ Multi-agent system created using gemini_flash (AdkApp & Runner pointing to main_agent)


## Step 8: Create User Session

In [8]:
# Create session directly in the runner's session store
user_id = "test-user-id"
app_name = getattr(runner, "app_name", "weather_assistant")

session = await runner.session_service.create_session(
    app_name=app_name, user_id=user_id
)

session_id = session.id if hasattr(session, "id") else session.get("id")
print(f"✅ Runner session created successfully: {session_id}")

✅ Runner session created successfully: 1ba5b60e-1da5-45a2-b4e6-7b04f6b5ce19


## Step 9: Agent Execution Function

In [9]:
import nest_asyncio

nest_asyncio.apply()


async def ask_weather_agent(query: str, session_id: str = None) -> str:
    """Query the weather agent using runner.run_async."""
    try:
        user_id = "test-user-id"
        app_name = getattr(runner, "app_name", "weather_assistant")

        # Fallback: create session if none provided
        if not session_id:
            current_session = await runner.session_service.create_session(
                app_name=app_name, user_id=user_id
            )
            session_id = (
                current_session.id
                if hasattr(current_session, "id")
                else current_session.get("id")
            )

        content = Content(role="user", parts=[Part(text=query)])
        final_text = None

        async for event in runner.run_async(
            user_id=user_id, session_id=session_id, new_message=content
        ):
            if hasattr(event, "is_final_response") and event.is_final_response():
                if (
                    hasattr(event, "content")
                    and hasattr(event.content, "parts")
                    and event.content.parts
                ):
                    final_text = event.content.parts[0].text
            elif (
                hasattr(event, "content")
                and hasattr(event.content, "parts")
                and event.content.parts
            ):
                final_text = event.content.parts[0].text

        return final_text or "No response received"
    except Exception as e:
        return f"Error: {str(e)}"


def query_weather(query: str, session_id: str = None) -> str:
    """Synchronous wrapper for Jupyter notebooks."""
    loop = asyncio.get_event_loop()
    return loop.run_until_complete(ask_weather_agent(query, session_id))


print("✅ Agent execution functions ready")

✅ Agent execution functions ready


## Step 10: Test & Output Events Demonstrating Sub-Agent Usage

In [10]:
from IPython.display import Markdown, display


def test_multi_agent(query: str):
    print(f"\n{'='*70}")
    print(f"📥 USER QUERY: {query}")
    print(f"{'='*70}")

    last_event = None
    try:
        for event in app.stream_query(
            user_id=user_id,
            session_id=session_id,
            message=query,
        ):
            last_event = event

            # Print routing/delegation events as they occur
            if isinstance(event, dict):
                author = event.get("author") or event.get("agent_name", "")
                actions = event.get("actions", {})
                if author:
                    print(f"🔄 [Event from Agent: {author}]")
                if actions and actions != {
                    "state_delta": {},
                    "artifact_delta": {},
                    "requested_auth_configs": {},
                    "requested_tool_confirmations": {},
                }:
                    print(f"   ⚙️ Tool/Sub-agent Actions: {actions}")

        # Display final synthesized answer
        if (
            last_event
            and isinstance(last_event, dict)
            and "content" in last_event
            and last_event["content"]
            and "parts" in last_event["content"]
            and len(last_event["content"]["parts"]) > 0
        ):
            print("\n📤 FINAL OUTPUT:")
            display(Markdown(last_event["content"]["parts"][0]["text"]))
        else:
            print("\n⚠️ No final content part returned.")
    except Exception as e:
        print(f"\n❌ Execution Error: {str(e)}")


# Test 1: Routes to Weather Sub-Agent
test_multi_agent("What is the current weather in Miami, Florida?")

# Test 2: Routes to Google Search Sub-Agent
test_multi_agent("Who won the most recent Super Bowl and what was the score?")


📥 USER QUERY: What is the current weather in Miami, Florida?


/Users/ridwan/.local/share/virtualenvs/docscan-wrm2pkcA/lib/python3.13/site-packages/vertexai/preview/reasoning_engines/templates/adk.py:975: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/Users/ridwan/.local/share/virtualenvs/docscan-wrm2pkcA/lib/python3.13/site-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()
App "default-app-name" can transfer between agents but has no context_cache_config. Every transfer swaps the system instruction and the tool set, so the request prefix changes and the whole prompt is re-sent uncached after e

🔄 [Event from Agent: main_agent]
🔄 [Event from Agent: main_agent]
   ⚙️ Tool/Sub-agent Actions: {'state_delta': {}, 'artifact_delta': {}, 'transfer_to_agent': 'weather_assistant', 'requested_auth_configs': {}, 'requested_tool_confirmations': {}}
🔄 [Event from Agent: weather_assistant]
🔄 [Event from Agent: weather_assistant]
🔄 [Event from Agent: weather_assistant]
🔄 [Event from Agent: weather_assistant]
🔄 [Event from Agent: weather_assistant]

📤 FINAL OUTPUT:


The current weather in Miami, Florida is 90°F with a chance of showers and thunderstorms. The wind is from the south at 6 to 10 mph. The heat index could reach as high as 106°F, so please take precautions for the heat.


📥 USER QUERY: Who won the most recent Super Bowl and what was the score?
🔄 [Event from Agent: weather_assistant]
🔄 [Event from Agent: weather_assistant]
   ⚙️ Tool/Sub-agent Actions: {'state_delta': {}, 'artifact_delta': {}, 'transfer_to_agent': 'main_agent', 'requested_auth_configs': {}, 'requested_tool_confirmations': {}}


Direct use of automatic function calling (AFC) in AsyncModels.generate_content is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message. Similarly, direct use of AFC in AsyncModels.generate_content_stream is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message_stream.


🔄 [Event from Agent: main_agent]
🔄 [Event from Agent: main_agent]
🔄 [Event from Agent: main_agent]

📤 FINAL OUTPUT:


The Seattle Seahawks won the most recent Super Bowl, Super Bowl LX, held on February 8, 2026. They defeated the New England Patriots with a score of 29-13.

## Step 11: Test Weather Sub Agent for Multiple US Cities

In [11]:
# Test dataset covering US regions, non-US locations, and edge cases
test_cities = [
    {
        "city": "New York, NY",
        "category": "US East Coast",
        "expected": "Success (Weather Agent)",
    },
    {
        "city": "Chicago, IL",
        "category": "US Midwest",
        "expected": "Success (Weather Agent)",
    },
    {
        "city": "Austin, TX",
        "category": "US South",
        "expected": "Success (Weather Agent)",
    },
    {
        "city": "Seattle, WA",
        "category": "US Pacific Northwest",
        "expected": "Success (Weather Agent)",
    },
    {
        "city": "Honolulu, HI",
        "category": "US Non-Contiguous",
        "expected": "Success (Weather Agent)",
    },
    {
        "city": "London, UK",
        "category": "International (Non-US)",
        "expected": "Callback Moderation Block",
    },
    {
        "city": "Tokyo, Japan",
        "category": "International (Non-US)",
        "expected": "Callback Moderation Block",
    },
]


def run_city_weather_test(city_info: dict):
    city_name = city_info["city"]
    category = city_info["category"]
    expected_outcome = city_info["expected"]

    query = f"What is the current weather forecast for {city_name}?"

    print(f"\n{'='*75}")
    print(f"📍 Testing: {city_name} [{category}]")
    print(f"🎯 Expected: {expected_outcome}")
    print(f"{'='*75}")

    last_event = None
    try:
        for event in app.stream_query(
            user_id=user_id,
            session_id=session_id,
            message=query,
        ):
            last_event = event

            # Track delegation and routing events
            if isinstance(event, dict):
                author = event.get("author") or event.get("agent_name")
                actions = event.get("actions")
                if author:
                    print(f"  🔄 [Routed to: {author}]")
                if actions and actions != {
                    "state_delta": {},
                    "artifact_delta": {},
                    "requested_auth_configs": {},
                    "requested_tool_confirmations": {},
                }:
                    print(f"  ⚙️ [Actions]: {actions}")

        # Render response
        if (
            last_event
            and isinstance(last_event, dict)
            and "content" in last_event
            and last_event["content"]
            and "parts" in last_event["content"]
            and len(last_event["content"]["parts"]) > 0
        ):
            output_text = last_event["content"]["parts"][0]["text"]
            print("  📤 Output:")
            display(Markdown(output_text))
        else:
            print("  ⚠️ No output generated.")

    except Exception as e:
        print(f"  ❌ Error during execution: {str(e)}")


# Run tests sequentially across all cities
for item in test_cities:
    run_city_weather_test(item)


📍 Testing: New York, NY [US East Coast]
🎯 Expected: Success (Weather Agent)
  🔄 [Routed to: main_agent]
  🔄 [Routed to: main_agent]
  ⚙️ [Actions]: {'state_delta': {}, 'artifact_delta': {}, 'transfer_to_agent': 'weather_assistant', 'requested_auth_configs': {}, 'requested_tool_confirmations': {}}
  🔄 [Routed to: weather_assistant]
  🔄 [Routed to: weather_assistant]
  🔄 [Routed to: weather_assistant]
  🔄 [Routed to: weather_assistant]
  🔄 [Routed to: weather_assistant]
  📤 Output:


The current weather in New York, NY is mostly sunny with a temperature of 80°F. The wind is from the northeast at 7 to 13 mph.


📍 Testing: Chicago, IL [US Midwest]
🎯 Expected: Success (Weather Agent)
  🔄 [Routed to: weather_assistant]
  🔄 [Routed to: weather_assistant]
  🔄 [Routed to: weather_assistant]
  🔄 [Routed to: weather_assistant]
  🔄 [Routed to: weather_assistant]
  📤 Output:


The current weather in Chicago, IL is 82°F with showers and thunderstorms likely, especially between noon and 3pm. There's a 60% chance of precipitation with new rainfall amounts less than a tenth of an inch possible. The wind is from the south at 5 to 10 mph.


📍 Testing: Austin, TX [US South]
🎯 Expected: Success (Weather Agent)
  🔄 [Routed to: weather_assistant]
  🔄 [Routed to: weather_assistant]
  🔄 [Routed to: weather_assistant]
  🔄 [Routed to: weather_assistant]
  🔄 [Routed to: weather_assistant]
  📤 Output:


...

The current weather in Honolulu, HI is sunny with a high near 89°F, though the heat index could reach as high as 96°F. There's a 20% chance of isolated rain showers after noon, with new rainfall amounts less than a tenth of an inch possible. The wind is from the east-southeast at 1 to 8 mph. Please be aware of the high heat index.


📍 Testing: London, UK [International (Non-US)]
🎯 Expected: Callback Moderation Block
  🔄 [Routed to: weather_assistant]
  🔄 [Routed to: weather_assistant]
  ⚙️ [Actions]: {'state_delta': {}, 'artifact_delta': {}, 'transfer_to_agent': 'main_agent', 'requested_auth_configs': {}, 'requested_tool_confirmations': {}}
  🔄 [Routed to: main_agent]
  🔄 [Routed to: main_agent]
  🔄 [Routed to: main_agent]
  📤 Output:


The current weather in London, UK, as of Friday, August 21, 2026, at 6:16 PM, is partly sunny with a temperature of 69°F (21°C). It feels like 75°F (24°C), and the humidity is around 42%. There is a 5% chance of rain.

The forecast for tonight, Friday, August 21st, indicates clear skies with a 15% chance of rain, and temperatures dropping to around 52°F (11°C).

Here's a brief outlook for the next few days:
*   **Saturday, August 22nd:** Sunny with a high of 70°F (21°C) and a low of 50°F (10°C), with a 10% chance of rain.
*   **Sunday, August 23rd:** Sunny with a high of 74°F (23°C) and a low of 51°F (11°C), with a 10% chance of rain.
*   **Monday, August 24th:** Partly sunny with a high of 70°F (21°C) and a low of 54°F (12°C), with a 10% chance of rain.


📍 Testing: Tokyo, Japan [International (Non-US)]
🎯 Expected: Callback Moderation Block
  🔄 [Routed to: main_agent]
  🔄 [Routed to: main_agent]
  🔄 [Routed to: main_agent]
  📤 Output:


The current weather in Tokyo, Japan, as of Saturday, August 22, 2026, at 2:16 AM local time, is cloudy with a temperature of 77°F (25°C). It feels like 79°F (26°C), with a humidity of around 88%. AccuWeather reports a RealFeel® Temperature of 83° (Very Warm) with 91% humidity and 91% cloud cover.

The forecast for Saturday, August 22, indicates cloudy conditions with periodic rain and a 40% chance of rain during the day. Temperatures are expected to range between 75°F (24°C) and 90°F (32°C), with humidity around 79%. There is also a risk of thunderstorms with a 30% chance, and scattered showers are possible, with some sources indicating up to 5-10mm of rain.

Looking ahead, Sunday, August 23, is also expected to be cloudy with a 40% chance of precipitation, and temperatures between 75°F (24°C) and 88°F (31°C).

## Step 12: Test Search Sub Agent for Multiple Questions

In [12]:
# Curated pool of city-specific search queries
city_search_questions = [
    "What are the top 3 historic landmarks to visit in Boston?",
    "What is the population and main industry of Austin, Texas?",
    "When was the Space Needle built in Seattle and how tall is it?",
    "What famous music festival takes place annually in Chicago's Grant Park?",
    "What are the best outdoor activities and national parks near Denver, Colorado?",
    "What is the story behind the French Quarter in New Orleans?",
    "Who is the current mayor of Miami and what are the major ports in the city?",
]

# Randomly select questions to test the search sub-agent
selected_search_questions = random.sample(city_search_questions, 3)

# Test 2: Routes to Google Search Sub-Agent with random city questions
print(f"{'#'*75}\n# Test 2: Routes to Google Search Sub-Agent (City Inquiries)\n{'#'*75}")

for question in selected_search_questions:
    test_multi_agent(question)

###########################################################################
# Test 2: Routes to Google Search Sub-Agent (City Inquiries)
###########################################################################

📥 USER QUERY: What are the best outdoor activities and national parks near Denver, Colorado?
🔄 [Event from Agent: main_agent]
🔄 [Event from Agent: main_agent]
🔄 [Event from Agent: main_agent]

📤 FINAL OUTPUT:


Denver, Colorado, serves as an excellent base for a wide array of outdoor activities and offers access to several stunning national parks.

The most prominent and accessible national park from Denver is **Rocky Mountain National Park (RMNP)**. Located approximately 70-80 miles northwest of the city, it's about a 1.5 to 2-hour drive. RMNP is open year-round and boasts over 350 miles of hiking trails, numerous lakes, and 77 mountains exceeding 12,000 feet. Visitors can enjoy wildlife viewing, scenic drives like Trail Ridge Road, and a variety of bird species. During peak season, a timed-entry reservation system is in place for popular areas.

Other national parks in Colorado that are a longer drive from Denver include:
*   **Great Sand Dunes National Park and Preserve** (approximately 4 hours away)
*   **Black Canyon of the Gunnison National Park** (about 4.5 to 5 hours away)
*   **Mesa Verde National Park** (7 to 7.5 hours away)

Denver and its surrounding areas offer a diverse range of outdoor activities:

*   **Hiking:** Popular options close to the city include Red Rocks Park Trail, Buffalo Trail, Lookout Mountain Trail, and the Maverick & Sawmill Loop Trail. Further out, recommended hikes include Mayflower Gulch Trail, Manitou Springs Incline, and trails in Roxborough State Park.
*   **Biking:** Denver has over 850 miles of paths, with trails like the Mile High Loop, Cherry Creek Trail, Clear Creek Trail, and Ralston Creek Trail being popular choices.
*   **Kayaking and Whitewater Rafting:** The South Platte River, Colorado River, and various lakes near Denver provide opportunities for kayaking. Whitewater rafting is popular on Clear Creek (just 30 minutes from Denver) and the Colorado River.
*   **Ziplining:** Adventure seekers can experience ziplining through the Rocky Mountains, with some of the longest and fastest ziplines in Colorado located just outside Denver.
*   **Local Parks and Gardens:**
    *   **Red Rocks Park and Amphitheatre** is great for hiking and biking.
    *   **Garden of the Gods** in Colorado Springs (1 to 1.5 hours drive south) is known for its stunning red rock formations.
    *   **Denver Botanic Gardens** offers a 24-acre oasis.
    *   **Rocky Mountain Arsenal National Wildlife Refuge**, located just 10 minutes from downtown, provides opportunities for birdwatching and viewing bison.
*   **Scenic Drives:** The Mount Blue Sky Scenic Byway, located 60 miles west of Denver, is the highest paved road in North America and offers breathtaking views.


📥 USER QUERY: When was the Space Needle built in Seattle and how tall is it?
🔄 [Event from Agent: main_agent]
🔄 [Event from Agent: main_agent]
🔄 [Event from Agent: main_agent]

📤 FINAL OUTPUT:


The Space Needle in Seattle was built for the 1962 World's Fair and officially opened on April 21, 1962. It stands at a height of 605 feet (184 meters).


📥 USER QUERY: What is the population and main industry of Austin, Texas?
🔄 [Event from Agent: main_agent]
🔄 [Event from Agent: main_agent]
🔄 [Event from Agent: main_agent]

📤 FINAL OUTPUT:


Austin, Texas, has a population that recently surpassed one million residents, with an estimated 1,011,743 in 2026. The city has experienced steady growth, increasing by 5.15% since the 2020 census.

The main industries in Austin are diverse, with technology, government, and education forming significant pillars of its economy. It is recognized as an important hub for high-tech, particularly in semiconductors and software. Other key sectors include healthcare, manufacturing, hospitality, and the creative arts. The presence of the state government and the University of Texas at Austin also plays a substantial role in the city's economic landscape.